# Robustness — Automated Labeling Rule (Blockscout tags)

The primary labeling rule uses public label lists plus an Etherscan check of every other funder of at least 50
LayerZero interactors (tracker A12). This notebook replaces the Etherscan step with a rule that has no
interactor threshold and no human step: every address that funded at least 2 addresses is labeled if
Blockscout's tag service gives it a service category tag (`build_blockscout_rule.py`, rule fixed before this
run). The public lists are the same. Everything else (features, gas provision trees, splits, the selected LightGBM
configuration) follows the primary pipeline; `sybil_pipeline.py` is not modified.

Comparison: LightGBM on the same 10 group-split seeds as `07`, one model seed, primary vs automated labels.

In [1]:
import os, json, warnings
os.chdir('../..')                   # run from the repo root so the pipeline's relative paths resolve
import numpy as np, pandas as pd
import sybil_pipeline as sp
warnings.filterwarnings('ignore'); pd.set_option('display.width', 200)
_, LGBM_SELECTED = sp.load_selected_params()
cur, _, anchors_cur = sp.build_master_df('./data', verbose=False)
_orig = sp.data_paths
def _alt(data_dir='./data'):
    p = _orig(data_dir); p['etherscan_services'] = 'review_support/blockscout_rule_sensitivity/blockscout_rule.csv'; return p
sp.data_paths = _alt
try:
    alt, _, anchors_alt = sp.build_master_df('./data', verbose=False)
finally:
    sp.data_paths = _orig
print(f'Labeled addresses in the network: primary {len(anchors_cur):,}, automated {len(anchors_alt):,} '
      f'(only primary {len(anchors_cur - anchors_alt)}, only automated {len(anchors_alt - anchors_cur)})')
b, a = cur.set_index('addr'), alt.set_index('addr').loc[cur['addr']]
changed = pd.Series(False, index=b.index)
for f in sp.FEATS:
    changed |= ~np.isclose(b[f].astype(float), a[f].astype(float), rtol=1e-9, atol=1e-12)
print(f'Wallets with any feature changed: {changed.sum():,} ({int(b.loc[changed, "sybil"].sum())} Sybil)')
print(pd.crosstab(b.loc[changed, 'category'], a.loc[changed, 'category']).to_string())

Labeled addresses in the network: primary 4,308, automated 4,311 (only primary 6, only automated 9)


Wallets with any feature changed: 3,477 (55 Sybil)
category   IxE  IxI  IxL
category                
IxE       1793    0   79
IxI          0  391    0
IxL        557  656    1


## Primary vs automated labels (LightGBM, 10 group splits, 1 model seed)

In [2]:
rows = []
for s in sp.SPLIT_SEEDS:
    for name, d in [('primary', cur), ('automated', alt)]:
        S = sp.make_splits(d, sp.FEATS, method='group', seed=s)
        rows.append({'split_seed': s, 'labels': name, **sp.lgbm_fit_eval(S, sp.FEATS, LGBM_SELECTED)})
res = pd.DataFrame(rows)
p = res.pivot(index='split_seed', columns='labels', values=['val_f1', 'test_f1', 'test_ap', 'test_auroc'])
for m in ['val_f1', 'test_f1', 'test_ap', 'test_auroc']:
    p[(m, 'diff')] = p[(m, 'automated')] - p[(m, 'primary')]
print(p.round(4).to_string())
summary = p.xs('diff', axis=1, level=1).agg(['mean', 'std'])
print('\nAutomated minus primary, mean and SD over 10 splits:'); print(summary.round(4).to_string())

              val_f1           test_f1           test_ap         test_auroc          val_f1 test_f1 test_ap test_auroc
labels     automated primary automated primary automated primary  automated primary    diff    diff    diff       diff
split_seed                                                                                                            
1             0.7168  0.7143    0.7108  0.6924    0.7665  0.7512     0.9696  0.9689  0.0025  0.0184  0.0153     0.0007
2             0.7040  0.7150    0.7186  0.7142    0.7653  0.7577     0.9680  0.9677 -0.0110  0.0044  0.0076     0.0003
3             0.7161  0.7080    0.7117  0.7109    0.7620  0.7632     0.9687  0.9694  0.0081  0.0008 -0.0012    -0.0007
4             0.7231  0.7007    0.6948  0.7076    0.7440  0.7559     0.9668  0.9685  0.0223 -0.0129 -0.0119    -0.0018
5             0.7106  0.7155    0.7054  0.7126    0.7550  0.7605     0.9643  0.9701 -0.0049 -0.0072 -0.0055    -0.0058
6             0.7172  0.7081    0.6996  0.6971  

In [3]:
sp.save_results([], 'review_blockscout_rule', out_dir='review_support/blockscout_rule_sensitivity', extra=dict(
    labeled_primary=len(anchors_cur), labeled_automated=len(anchors_alt),
    only_primary=sorted(anchors_cur - anchors_alt), only_automated=sorted(anchors_alt - anchors_cur),
    wallets_changed=int(changed.sum()), per_split=res.to_dict('records'),
    diff_mean=summary.loc['mean'].to_dict(), diff_sd=summary.loc['std'].to_dict(), params=LGBM_SELECTED))

Saved review_support/blockscout_rule_sensitivity/review_blockscout_rule.json
